# Train SMD Component Detector & Export TFLite

Dataset: 7791 images, 4 classes (Resistor, Diode, Transistor, Condensator)
Model: YOLOv5n — fast & small, exports to TFLite

**Single-click: Run this cell → wait 10-15 mins → download .tflite**

In [ ]:
# Step 1: Install dependencies
!pip install -q roboflow ultralytics onnx onnxruntime
print("Dependencies installed")

# Step 2: Download dataset in YOLOv5 format
from roboflow import Roboflow
rf = Roboflow(api_key="aLtYgQ8E7uKE7s3FNO20")
project = rf.workspace("dainius").project("smdcomponents")
dataset = project.version(6).download("yolov5")
print(f"Dataset: {dataset.location}")

# Check data.yaml location
import os
yaml_path = os.path.join(dataset.location, "data.yaml")
if not os.path.exists(yaml_path):
    # Try alternate path structure
    for root, dirs, files in os.walk(dataset.location):
        for f in files:
            if f.endswith('.yaml'):
                yaml_path = os.path.join(root, f)
                print(f"Found data.yaml: {yaml_path}")
                break
print(f"Using data.yaml: {yaml_path}")
print(f"Exists: {os.path.exists(yaml_path)}")

# Step 3: Train YOLOv5n
!yolo detect train \
  data="{yaml_path}" \
  model=yolov5n.pt \
  epochs=20 \
  imgsz=416 \
  batch=32 \
  name=smd_detector \
  exist_ok=True

print()
print("Training complete!")

# Step 4: Export to TFLite
from ultralytics import YOLO
import glob
pt_files = glob.glob("runs/detect/smd_detector/**/best.pt", recursive=True)
if not pt_files:
    pt_files = glob.glob("runs/detect/**/best.pt", recursive=True)
if not pt_files:
    pt_files = ["runs/detect/smd_detector/weights/best.pt"]
pt_path = pt_files[0]
print(f"Model: {pt_path}")
print(f"Exists: {os.path.exists(pt_path)}")

if os.path.exists(pt_path):
    model = YOLO(pt_path)
    model.export(format="tflite")
    print()
    print("TFLite model exported!")
    
    # Find and download TFLite
    tfl = glob.glob("runs/detect/smd_detector/**/*.tflite", recursive=True)
    if tfl:
        from google.colab import files
        files.download(tfl[0])
        print(f"Downloaded: {tfl[0]}")
else:
    print("Model file not found — training may have failed")